# 강의 04 · 실습 8 — 이미지 생성 파이프라인 · (3) 변형

## 1. 문제상황

- 운영팀 디자이너는 다시 뽑을 때마다 지시문을 처음부터 새로 씁니다.
- 무엇을 바꿔서 결과가 좋아졌는지 알 수 없어서, 같은 개선을 다음 주제에 다시 쓰지 못합니다.
- 팀은 지시문 템플릿을 두고, 다시 뽑을 때는 템플릿의 변수 하나만 바꾸기로 정했습니다.
- 이번에 바꾸는 변수는 조명 묘사 구절 하나입니다. 기준 템플릿 v1에 조명의 방향과 색온도를 적으라는 구절을 더한 것이 v2입니다.

## 2. 문제와 목표

- **문제**: 다시 뽑을 때 무엇을 바꿨는지가 남지 않습니다. 바꾼 변수가 하나로 고정되어야 결과의 차이를 그 변수 탓으로 돌릴 수 있습니다.
- **목표**: 주제를 입력하면 프로그램이 상태에 든 템플릿으로 지시문을 쓰고 이미지를 뽑고, 사람이 「재설계」라고 답하면 템플릿을 v1에서 v2로 바꾼 뒤 지시문 설계부터 다시 도는 처리 흐름을 만듭니다.
    - 템플릿 두 벌: v1과, v1에 조명 구절 하나(광원의 방향과 색온도)를 더한 v2입니다. 문면은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
    - 상태 키 다섯 개: 주제, 템플릿, 지시문, 이미지 경로 목록, 판정입니다.
    - 노드 네 개: design·generate·review에 템플릿을 v2로 바꾸는 revise가 더해집니다. 사람의 답(「재설계」 한 번, 「확정」 한 번)은 코드에 대본으로 미리 정해 넣습니다.
- **목표 달성 여부의 판정 기준**
    - 「재설계」라고 답한 뒤 두 번째 지시문에 광원의 방향과 색온도 묘사가 들어갑니다.
    - 후보가 2장이 된 채 다시 멈춥니다.
    - 「확정」이라고 답하면 END에 도달하는 것을 실행 결과에서 확인합니다.


## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec04_ex08_s3_diagram.svg)

## 4. 단계별 요구사항

1. **상태를 정의합니다.**
    - 주제(`topic`), 지시문 템플릿(`template`), 이미지 지시문(`prompt`), 생성한 이미지 경로 목록(`images`, `add` 리듀서), 사람의 판정(`verdict`) 키 다섯 개를 가지는 상태를 선언합니다.
2. **지시문 설계 노드를 만듭니다.**
    - design 노드는 상태의 `template` 뒤에 주제를 붙여 모델을 한 번 호출하고, 받은 지시문을 `prompt` 키에 씁니다.
    - 고정된 템플릿 상수를 직접 쓰지 않습니다.
3. **이미지 생성 노드를 만듭니다.**
    - generate 노드는 상태의 지시문을 `paint`로 저장하고, 저장된 경로 하나를 담은 목록을 `images` 키에 돌려줍니다.
4. **평가 노드를 만듭니다.**
    - review 노드는 `interrupt()`로 멈추고, 질문과 지금까지의 후보 경로 목록을 사람에게 보냅니다.
    - 사람의 답을 문자열로 `verdict` 키에 씁니다.
5. **템플릿 개정 노드를 만듭니다.**
    - revise 노드는 상태의 `template` 키를 `SPEC_V2`로 바꿉니다.
    - 바뀐 것은 조명 구절 하나뿐입니다.
    - 값은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
6. **그래프에 노드를 등록합니다.**
    - 네 노드를 이름과 함께 그래프에 등록합니다.
7. **엣지를 연결합니다.**
    - START → design → generate → review를 고정 엣지로 연결하고, review 뒤에는 조건부 엣지를 추가합니다.
    - 판정이 「재설계」이면 revise로, 그 밖에는 END로 갑니다.
    - revise 뒤에는 design을 고정 엣지로 연결합니다.
8. **그래프를 컴파일하고 실행합니다.**
    - 체크포인터를 달아 컴파일하고, 주제 「비 내리는 밤 서울 골목의 LP 바 창가」와 템플릿 v1을 넣어 실행합니다.
    - 멈춘 뒤 `Command(resume="재설계")`로 되돌리고, 두 번째 지시문에 조명 구절이 반영된 것과 후보가 2장이 된 것을 확인한 뒤 `Command(resume="확정")`으로 종료합니다.
    - 실행 셀은 멈춘 지점을 「next = …」 줄로, 재개 뒤 판정을 「verdict = …」 줄로, 끝에 생성 호출 횟수를 출력하고, revise 노드는 「[revise] 템플릿을 v2로 바꿉니다」 줄을 출력합니다.

## 5. 코드 골격 — LangGraph 5단

랭그래프(LangGraph)로 그래프를 세우는 순서는 다음 다섯 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 다섯 단계와 하나씩 대응합니다. 이미지 모델을 부르는 노드와 `interrupt()`는 새 단계가 아니라 ② 노드 함수와 ⑤ 실행 단계 안에 들어갑니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 상태 정의 | 노드들이 함께 읽고 쓸 키를 선언합니다 | `class ImageState(TypedDict)`, `Annotated[list, add]` | 1 |
| ② 노드 함수 정의 | 상태를 받아 바뀐 키만 돌려주는 함수를 만듭니다 | `def design(state) -> dict`, `paint()`, `interrupt()` | 2, 3, 4, 5 |
| ③ 그래프 빌더 생성과 노드 등록 | 빈 그래프를 열고 함수에 이름을 붙여 등록합니다 | `StateGraph(ImageState)`, `add_node` | 6 |
| ④ 엣지 연결 | 노드 사이의 순서와 분기를 정합니다 | `add_edge`, `add_conditional_edges` | 7 |
| ⑤ 컴파일과 실행 | 체크포인터를 달아 컴파일하고, 멈춘 지점에서 사람의 답으로 이어 갑니다 | `compile(checkpointer=…)`, `invoke`, `Command(resume=…)` | 8 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 텍스트 모델과 이미지 모델을 준비합니다. 이미지 모델을 부르는 함수 `paint`도 여기서 정의합니다.

- API 키와 자격증명은 `.env` 파일에서 읽습니다. `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다.
- `.env` 파일에는 다음 네 줄이 있어야 합니다. 값은 각자 발급받은 것을 넣습니다.

```
OPENAI_API_KEY=발급받은_키
GOOGLE_APPLICATION_CREDENTIALS=서비스_계정_키_파일의_경로
VERTEX_PROJECT=프로젝트_이름
VERTEX_LOCATION=리전_이름
```

- `paint` 호출 1회가 이미지 1장이고, 호출마다 비용이 듭니다. 생성한 이미지는 노트북 옆의 `out_images` 폴더에 저장됩니다.
- `show`는 후보 이미지 경로 목록을 화면에 표시하는 보조 함수입니다.
- 생성 호출 횟수는 전역 카운터 `paint_calls`로 셉니다.

In [ ]:
import os
import time
from operator import add
from pathlib import Path

from dotenv import load_dotenv, find_dotenv
from typing import Annotated, TypedDict

from IPython.display import Image, display
from google import genai
from google.genai import types
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

load_dotenv(find_dotenv(usecwd=True))
for key in ("OPENAI_API_KEY", "GOOGLE_APPLICATION_CREDENTIALS", "VERTEX_PROJECT", "VERTEX_LOCATION"):
    if not os.environ.get(key):
        raise SystemExit(f"agentic-ai 폴더의 .env 파일에 {key} 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
client = genai.Client(vertexai=True, project=os.environ["VERTEX_PROJECT"], location=os.environ["VERTEX_LOCATION"])
IMG_MODEL = "gemini-3.1-flash-lite-image"
OUT_DIR = Path("out_images")
OUT_DIR.mkdir(exist_ok=True)
paint_calls = 0


def paint(prompt: str, tag: str) -> str:
    """지시문을 이미지 모델에 보내 이미지 파일을 저장하고 경로를 돌려준다. 호출 1회 = 이미지 1장 = 비용 발생."""
    global paint_calls
    for attempt in range(4):
        try:
            res = client.models.generate_content(
                model=IMG_MODEL, contents=prompt,
                config=types.GenerateContentConfig(response_modalities=["IMAGE", "TEXT"]))
            break
        except Exception as e:
            if "429" in str(e) and attempt < 3:
                print(f"    [paint] 분당 한도 초과 — {30 * (attempt + 1)}초 뒤 다시 부릅니다")
                time.sleep(30 * (attempt + 1))
                continue
            raise
    paint_calls += 1
    part = [p for p in res.candidates[0].content.parts if p.inline_data][0].inline_data
    ext = "png" if "png" in part.mime_type else "jpg"
    path = OUT_DIR / f"{tag}_{paint_calls:02d}.{ext}"
    path.write_bytes(part.data)
    print(f"    [paint] {path.as_posix()} ({len(part.data)} bytes)")
    return path.as_posix()


def show(paths: list) -> None:
    """후보 이미지 경로 목록을 화면에 차례로 표시한다."""
    for i, p in enumerate(paths, 1):
        print(f"    후보 {i}: {p}")
        display(Image(filename=p, width=320))


print("모델 준비를 마쳤습니다. 이미지 저장 폴더:", OUT_DIR)

# 주어진 자료 — 지시문 템플릿 두 벌 (주제 문장을 뒤에 이어 붙여 언어 모델에 넣는다)
SPEC_V1 = ("다음 주제로 이미지 생성 지시문을 한 문단으로 쓴다. "
           "장면·조명·화각을 포함한다. 주제: ")
SPEC_V2 = ("다음 주제로 이미지 생성 지시문을 한 문단으로 쓴다. "
           "장면·조명·화각을 포함한다. 조명은 광원의 방향과 색온도를 한 구절로 적는다. 주제: ")


### 단계 ① — 상태 정의 (요구사항 1)

주제·지시문·이미지 경로 목록·판정 키 네 개에 `template` 키가 더해집니다. 템플릿을 상태에 두면 노드가 상태를 바꾸는 것만으로 다음 설계의 조건이 바뀝니다.

In [ ]:
# 여기에 단계 ①(상태 정의: template 키 포함)을 작성합니다.

### 단계 ② — 노드 함수 정의 (요구사항 2, 3, 4, 5)

- design 노드는 상수 `SPEC_V1`이 아니라 상태의 `template` 키를 읽습니다.
- revise 노드는 모델을 부르지 않습니다. 상태의 `template` 키를 `SPEC_V2`로 바꾸는 일만 합니다.
- `SPEC_V2`는 `SPEC_V1`에 조명 구절 하나를 더한 것입니다. 나머지 문면은 같습니다.

In [ ]:
# 여기에 단계 ②(노드 함수 네 개 정의)을 작성합니다.

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 6)

노드가 네 개가 되었습니다.

In [ ]:
# 여기에 단계 ③(그래프 빌더 생성과 노드 등록)을 작성합니다.

### 단계 ④ — 엣지 연결 (요구사항 7)

조건부 엣지의 매핑에 `revise`가 들어가고, revise → design 고정 엣지가 되돌림을 완성합니다.

In [ ]:
# 여기에 단계 ④(엣지 연결과 판단 함수)을 작성합니다.

### 단계 ⑤ — 컴파일과 실행 (요구사항 8)

초기 상태에 `template`으로 `SPEC_V1`을 넣습니다. 컴파일, 「재설계」 재개, 「확정」 재개의 세 부분이 모두 이 한 단계에 속합니다.

실행 설정은 `config = {"configurable": {"thread_id": "design-v1v2"}}`이고, 멈춘 지점은 `graph.get_state(config).next`, 사람에게 간 내용은 `invoke` 반환값의 `out["__interrupt__"][0].value`에서 읽습니다.


In [ ]:
# 여기에 단계 ⑤(컴파일과 실행: 멈춤 확인, 재설계로 되돌림, 확정으로 종료)을 작성합니다.

## 7. 실행 결과 확인

위 실행 결과에서 다음 네 가지를 확인합니다.

1. 1차 실행에서 `next = ('review',)`가 출력되고 후보 1장이 표시됩니다. 지시문은 템플릿 v1로 쓴 것입니다.
2. 재개 A에서 `[revise] 템플릿을 v2로 바꿉니다` 줄이 먼저 출력되고, `template` 키가 v2로 바뀐 채 design이 다시 실행됩니다. 새 지시문에 광원의 방향이나 색온도 묘사가 들어 있습니다.
3. 재개 A의 끝에서 후보가 2장이 된 채 review에서 다시 멈춥니다.
4. 재개 B에서 `verdict = 확정`, `next = ()`가 출력되고 생성 호출은 2회입니다. 바뀐 변수는 조명 구절 하나뿐이므로, 두 후보의 차이는 그 구절 탓으로 돌릴 수 있습니다.